# Phase 4 — Feature Engineering

## Real Estate Investment Advisor

This notebook creates meaningful derived features from the cleaned housing dataset.

### Objectives

- Create a reliable Price per SqFt feature
- Create Amenity Density Score
- Create useful property-level features
- Validate engineered features
- Save the feature-engineered dataset

In [24]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [25]:
cleaned_path = "../data/processed/india_housing_prices_cleaned.csv"

df = pd.read_csv(cleaned_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (250000, 23)


In [26]:
df.head()

,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,Furnished,22,1,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,Unfurnished,21,20,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,Semi-furnished,19,27,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,Furnished,21,26,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,Semi-furnished,3,2,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  str    
 2   City                            250000 non-null  str    
 3   Locality                        250000 non-null  str    
 4   Property_Type                   250000 non-null  str    
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  str    
 11  Floor_No                        250000 non-null  int64  
 12  Total_Floors               

## 1. Calculated Price per SqFt

The source `Price_per_SqFt` field was found to be inconsistent with property price and size.

A reliable price-per-square-foot feature is therefore calculated using:

Price per SqFt = (Price in Lakhs × 100000) / Size in SqFt

In [28]:
df["Calculated_Price_per_SqFt"] = (
    df["Price_in_Lakhs"] * 100000
) / df["Size_in_SqFt"]

print("Calculated_Price_per_SqFt created.")

Calculated_Price_per_SqFt created.


In [29]:
df[
    [
        "Price_in_Lakhs",
        "Size_in_SqFt",
        "Price_per_SqFt",
        "Calculated_Price_per_SqFt"
    ]
].head(10)

,Price_in_Lakhs,Size_in_SqFt,Price_per_SqFt,Calculated_Price_per_SqFt
0,489.76,4740,0.10,10332.489451
1,195.52,2364,0.08,8270.727580
2,183.79,3642,0.05,5046.403075
3,300.29,2741,0.11,10955.490697
4,182.90,4823,0.04,3792.245490
5,135.28,3500,0.04,3865.142857
6,318.12,4826,0.07,6591.794447
7,141.39,4252,0.03,3325.258702
8,189.16,2678,0.07,7063.480209
9,187.42,1393,0.13,13454.414932


In [30]:
df["Calculated_Price_per_SqFt"].describe()

count    250000.000000
mean      13058.281776
std       13071.850480
min         202.247191
25%        4802.839710
50%        9244.747594
75%       15987.388517
max       99182.000000
Name: Calculated_Price_per_SqFt, dtype: float64

## 2. Amenity Density Score

The `Amenities` field contains comma-separated amenities.

Amenity Density Score represents the number of amenities available for each property.

In [31]:
df["Amenity_Density_Score"] = (
    df["Amenities"]
    .fillna("")
    .apply(
        lambda x: len(
            [amenity.strip() for amenity in x.split(",") if amenity.strip()]
        )
    )
)

print("Amenity_Density_Score created.")

Amenity_Density_Score created.


In [32]:
df["Amenity_Density_Score"].value_counts().sort_index()

Amenity_Density_Score
1    50106
2    49806
3    49862
4    50362
5    49864
Name: count, dtype: int64

In [33]:
df["Amenity_Density_Score"].describe()

count    250000.000000
mean          3.000288
std           1.414284
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           5.000000
Name: Amenity_Density_Score, dtype: float64

## 3. Price per Bedroom

Price per Bedroom provides a normalized measure of property price relative to the number of bedrooms.

In [34]:
df["Price_per_BHK"] = (
    df["Price_in_Lakhs"] / df["BHK"]
)

print("Price_per_BHK created.")

Price_per_BHK created.


## 4. Size per Bedroom

Size per BHK represents the approximate property area available per bedroom.

In [35]:
df["Size_per_BHK"] = (
    df["Size_in_SqFt"] / df["BHK"]
)

print("Size_per_BHK created.")

Size_per_BHK created.


## 5. Property Age Category

Properties are grouped into broad age categories for analytical purposes.

In [36]:
df["Property_Age_Category"] = pd.cut(
    df["Age_of_Property"],
    bins=[-1, 5, 10, 20, 30, np.inf],
    labels=[
        "New",
        "Recent",
        "Moderate",
        "Old",
        "Very_Old"
    ]
)

print("Property_Age_Category created.")

Property_Age_Category created.


In [37]:
df["Property_Age_Category"].value_counts().sort_index()

Property_Age_Category
New         29566
Recent      36772
Moderate    73426
Old         73656
Very_Old    36580
Name: count, dtype: int64

## 6. Floor Consistency Indicator

A diagnostic indicator is created to identify records where the reported floor number exceeds the reported total floors.

The original values are preserved.

In [38]:
df["Floor_Consistency"] = np.where(
    df["Floor_No"] <= df["Total_Floors"],
    "Consistent",
    "Inconsistent"
)

print(df["Floor_Consistency"].value_counts())

Floor_Consistency
Consistent      133696
Inconsistent    116304
Name: count, dtype: int64


In [39]:
engineered_columns = [
    "Calculated_Price_per_SqFt",
    "Amenity_Density_Score",
    "Price_per_BHK",
    "Size_per_BHK",
    "Property_Age_Category",
    "Floor_Consistency"
]

df[engineered_columns].head(10)

,Calculated_Price_per_SqFt,Amenity_Density_Score,Price_per_BHK,Size_per_BHK,Property_Age_Category,Floor_Consistency
0,10332.489451,5,489.760000,4740.000000,Very_Old,Inconsistent
1,8270.727580,5,65.173333,788.000000,Moderate,Inconsistent
2,5046.403075,4,91.895000,1821.000000,Old,Consistent
3,10955.490697,5,150.145000,1370.500000,Very_Old,Consistent
4,3792.245490,5,45.725000,1205.750000,Old,Inconsistent
5,3865.142857,2,33.820000,875.000000,New,Inconsistent
6,6591.794447,3,106.040000,1608.666667,Recent,Inconsistent
7,3325.258702,4,28.278000,850.400000,New,Inconsistent
8,7063.480209,3,47.290000,669.500000,Old,Inconsistent
9,13454.414932,4,93.710000,696.500000,Moderate,Consistent


In [40]:
df[
    [
        "Calculated_Price_per_SqFt",
        "Amenity_Density_Score",
        "Price_per_BHK",
        "Size_per_BHK"
    ]
].describe()

,Calculated_Price_per_SqFt,Amenity_Density_Score,Price_per_BHK,Size_per_BHK
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,13058.281776,3.000288,116.410245,1256.992428
std,13071.850480,1.414284,106.535444,1065.087711
min,202.247191,1.000000,2.006000,100.000000
25%,4802.839710,2.000000,43.987667,542.000000
50%,9244.747594,3.000000,84.885000,918.000000
75%,15987.388517,4.000000,147.925417,1563.500000
max,99182.000000,5.000000,499.970000,5000.000000


In [41]:
print(
    "Invalid Calculated Price per SqFt:",
    (df["Calculated_Price_per_SqFt"] <= 0).sum()
)

print(
    "Invalid Amenity Density:",
    (df["Amenity_Density_Score"] <= 0).sum()
)

print(
    "Invalid Price per BHK:",
    (df["Price_per_BHK"] <= 0).sum()
)

print(
    "Invalid Size per BHK:",
    (df["Size_per_BHK"] <= 0).sum()
)

Invalid Calculated Price per SqFt: 0
Invalid Amenity Density: 0
Invalid Price per BHK: 0
Invalid Size per BHK: 0


In [42]:
df["Price_per_SqFt"] = df["Calculated_Price_per_SqFt"]

df.drop(
    columns=["Calculated_Price_per_SqFt"],
    inplace=True
)

print("Price_per_SqFt replaced with the calculated value.")

Price_per_SqFt replaced with the calculated value.


In [43]:
df[
    [
        "Price_in_Lakhs",
        "Size_in_SqFt",
        "Price_per_SqFt"
    ]
].head(10)

,Price_in_Lakhs,Size_in_SqFt,Price_per_SqFt
0,489.76,4740,10332.489451
1,195.52,2364,8270.727580
2,183.79,3642,5046.403075
3,300.29,2741,10955.490697
4,182.90,4823,3792.245490
5,135.28,3500,3865.142857
6,318.12,4826,6591.794447
7,141.39,4252,3325.258702
8,189.16,2678,7063.480209
9,187.42,1393,13454.414932


In [44]:
print("Total columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

Total columns: 28

Columns:
['ID', 'State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Amenity_Density_Score', 'Price_per_BHK', 'Size_per_BHK', 'Property_Age_Category', 'Floor_Consistency']


In [45]:
engineered_path = "../data/processed/india_housing_prices_engineered.csv"

df.to_csv(engineered_path, index=False)

print("Feature-engineered dataset saved successfully.")
print(engineered_path)

Feature-engineered dataset saved successfully.
../data/processed/india_housing_prices_engineered.csv


In [46]:
engineered_df = pd.read_csv(engineered_path)

print("Shape:", engineered_df.shape)
print("Missing values:", engineered_df.isnull().sum().sum())
print("Duplicate rows:", engineered_df.duplicated().sum())

Shape: (250000, 28)
Missing values: 0
Duplicate rows: 0


# Phase 4 — Feature Engineering Summary

The cleaned housing dataset was enhanced with domain-relevant derived features.

### Engineered Features

1. `Price_per_SqFt`
   - Recalculated from property price and size.
   - Replaces the unreliable source value.

2. `Amenity_Density_Score`
   - Number of amenities associated with the property.
   - Range: 1–5.

3. `Price_per_BHK`
   - Price in lakhs divided by number of bedrooms.

4. `Size_per_BHK`
   - Property area divided by number of bedrooms.

5. `Property_Age_Category`
   - Groups properties into New, Recent, Moderate, Old and Very_Old.

6. `Floor_Consistency`
   - Indicates whether Floor_No is less than or equal to Total_Floors.
   - Original floor values are preserved.

### Output

The feature-engineered dataset is saved as:

`data/processed/india_housing_prices_engineered.csv`